# Interactive object selection with SAM3 point expansion

Two steps on a hyperspectral session:

1. **Object selection** - click positive (object) and negative (background) points on one
   frame and let SAM3 expand them into an object mask.
2. **Propagation** - seed that mask into `SAM3MaskPropagation` and track it across 100 frames.

Step 1 uses the single-frame, re-promptable `SAM3PointExpansion` node; step 2 reuses the
streaming mask-propagation node. Needs the `cuvis-ai-sam3` plugin and a GPU.

In [ ]:
# Colab bootstrap - no-op when running locally
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q cuvis-ai "cuvis-ai-dataloader[cu3s,coco]" cuvis-ai-sam3
    !which ffmpeg dot >/dev/null || echo "WARN: ffmpeg/dot missing on this runtime"

    import torch

    if not torch.cuda.is_available():
        print("WARNING: No GPU detected. SAM3 on CPU is very slow.")

In [ ]:
# ruff: noqa: E402
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from cuvis_ai_dataloader.data import Cu3sDataModule
from IPython.display import Video, display
from loguru import logger

from cuvis_ai.node.anomaly_visualization import TrackingOverlayNode
from cuvis_ai.node.channel_selector import CIETristimulusRGBSelector
from cuvis_ai.node.data import CU3SDataNode
from cuvis_ai.node.json_file import CocoTrackMaskWriter
from cuvis_ai.node.prompts import MaskPrompt, PointPrompt
from cuvis_ai.node.video import ToVideoNode
from cuvis_ai_core.data.public_datasets import PublicDatasets
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import Predictor
from cuvis_ai_core.utils.node_registry import NodeRegistry

In [ ]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
logger.info(f"Using device {device}")

## 1 - Fetch the dataset

In [ ]:
dataset_dir = Path("/content/data") if IN_COLAB else Path("../../data")

_ = PublicDatasets.download_dataset(
    "demo_object_tracking",
    download_path=str(dataset_dir),
    force=False,
)

## 2 - Configuration

In [ ]:
PROCESSING_MODE = "SpectralRadiance"
PROMPT_FRAME_ID = 11  # frame to select the object on
NUM_FRAMES = 100  # frames to propagate the mask across
OBJECT_ID = 1  # single-object expansion
FRAME_RATE = 15.0

# Object selection as fractions of the frame (resolved to pixels once H, W are known).
# A positive point on the person in the centre; add negatives to push background out.
POSITIVE_POINTS_FRAC = [(0.5, 0.5)]
NEGATIVE_POINTS_FRAC = []  # e.g. [(0.05, 0.05)]

output_dir = Path("./output/point_expansion")
output_dir.mkdir(parents=True, exist_ok=True)
expansion_json = output_dir / "expansion_mask.json"  # COCO mask written by step A
mask_video_path = output_dir / "mask_propagation.mp4"  # propagation overlay from step B

cu3s_path = (
    dataset_dir
    / "XMR_Demo_Object_Tracking"
    / "measurements"
    / "cu3s"
    / "2026_04_15_16_28_10"
    / "Auto_000.cu3s"
)
print(f"CU3S:           {cu3s_path}")
print(f"Prompt frame:   {PROMPT_FRAME_ID}")
print(f"Expansion JSON: {expansion_json}")
print(f"Mask video:     {mask_video_path}")

## 3 - Load the SAM3 plugin

In [ ]:
PLUGINS_YAML = Path("../../cuvis_ai/configs/plugins/sam3.yaml")

registry = NodeRegistry()
registry.register_plugin(str(PLUGINS_YAML))
SAM3PointExpansion = registry.get("cuvis_ai_sam3.node.SAM3PointExpansion")
SAM3MaskPropagation = registry.get("cuvis_ai_sam3.node.SAM3MaskPropagation")
logger.success("SAM3 nodes loaded: {}, {}", SAM3PointExpansion, SAM3MaskPropagation)

## 4 - Step A: expand points into an object mask

Preview the prompt frame and resolve the click points to pixels. Locally you can swap in an
interactive picker (`%matplotlib widget` + `plt.ginput`); the default centre positive point
lets the notebook also run unattended on Colab.

In [ ]:
# Preview the prompt frame so we can place points on the object.
preview_dm = Cu3sDataModule(
    cu3s_file_path=str(cu3s_path),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    measurement_indices=[PROMPT_FRAME_ID],
)
preview_pipeline = CuvisPipeline("PointExpansion_Preview")
pv_cu3s = CU3SDataNode(name="cu3s_data")
pv_rgb = CIETristimulusRGBSelector(name="true_rgb")
preview_pipeline.connect(
    (pv_cu3s.outputs.cube, pv_rgb.cube),
    (pv_cu3s.outputs.wavelengths, pv_rgb.wavelengths),
)
preview_pipeline.to(device)
pv_out = Predictor(pipeline=preview_pipeline, datamodule=preview_dm).predict(collect_outputs=True)
rgb_frame = pv_out[0][("true_rgb", "rgb_image")][0].cpu().numpy()
frame_h, frame_w = rgb_frame.shape[:2]

positive_points = [(fx * frame_w, fy * frame_h) for fx, fy in POSITIVE_POINTS_FRAC]
negative_points = [(fx * frame_w, fy * frame_h) for fx, fy in NEGATIVE_POINTS_FRAC]

plt.figure(figsize=(9, 7))
plt.imshow(rgb_frame)
for px, py in positive_points:
    plt.scatter([px], [py], c="#2ecc71", marker="+", s=240, linewidths=3)
for px, py in negative_points:
    plt.scatter([px], [py], c="#e74c3c", marker="_", s=240, linewidths=3)
plt.title(f"Prompt frame {PROMPT_FRAME_ID} ({frame_w}x{frame_h}) - green + object, red - bg")
plt.axis("off")
plt.show()

In [ ]:
# Build the point-expansion pipeline: cube -> RGB -> SAM3PointExpansion -> COCO mask writer.
points = [(px, py, "positive") for px, py in positive_points]
points += [(px, py, "negative") for px, py in negative_points]

expansion_pipeline = CuvisPipeline("PointExpansion")
ex_cu3s = CU3SDataNode(name="cu3s_data")
ex_rgb = CIETristimulusRGBSelector(name="true_rgb")
point_prompt = PointPrompt(points=points, prompt_frame_id=PROMPT_FRAME_ID, name="point_prompt")
sam3_point = SAM3PointExpansion(prompt_obj_id=OBJECT_ID, name="sam3_point_expansion")
expansion_json_writer = CocoTrackMaskWriter(
    output_json_path=str(expansion_json),
    default_category_name="person",
    name="expansion_json",
)
expansion_pipeline.connect(
    (ex_cu3s.outputs.cube, ex_rgb.cube),
    (ex_cu3s.outputs.wavelengths, ex_rgb.wavelengths),
    (ex_rgb.rgb_image, sam3_point.inputs.rgb_frame),
    (ex_cu3s.outputs.mesu_index, sam3_point.inputs.frame_id),
    (ex_cu3s.outputs.mesu_index, point_prompt.inputs.frame_id),
    (point_prompt.outputs.points, sam3_point.inputs.points),
    (ex_cu3s.outputs.mesu_index, expansion_json_writer.inputs.frame_id),
    (sam3_point.outputs.mask, expansion_json_writer.inputs.mask),
    (sam3_point.outputs.object_ids, expansion_json_writer.inputs.object_ids),
    (sam3_point.outputs.detection_scores, expansion_json_writer.inputs.detection_scores),
)

expansion_dm = Cu3sDataModule(
    cu3s_file_path=str(cu3s_path),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    measurement_indices=[PROMPT_FRAME_ID],
)
expansion_pipeline.to(device)
ex_out = Predictor(pipeline=expansion_pipeline, datamodule=expansion_dm).predict(
    collect_outputs=True
)

mask = ex_out[0][("sam3_point_expansion", "mask")][0].cpu().numpy()
logger.success("Mask pixels: {} | COCO JSON: {}", int((mask > 0).sum()), expansion_json)

overlay = rgb_frame.copy()
overlay[mask > 0] = 0.5 * overlay[mask > 0] + 0.5 * np.array([0.0, 1.0, 1.0])
plt.figure(figsize=(9, 7))
plt.imshow(overlay)
for px, py in positive_points:
    plt.scatter([px], [py], c="#2ecc71", marker="+", s=240, linewidths=3)
plt.title("Expanded object mask (cyan)")
plt.axis("off")
plt.show()

## 5 - Step B: propagate the mask for 100 frames

`MaskPrompt` reads the COCO mask written above and seeds it on the prompt frame;
`SAM3MaskPropagation` then tracks it forward. The spec is `object_id:detection_id@frame_id`
(detection_id `1` is the single object written on the prompt frame).

In [ ]:
MASK_PROMPT_SPEC = f"{OBJECT_ID}:1@{PROMPT_FRAME_ID}"

prop_pipeline = CuvisPipeline("MaskPropagation")
pr_cu3s = CU3SDataNode(name="cu3s_data")
pr_rgb = CIETristimulusRGBSelector(name="true_rgb")
mask_prompt = MaskPrompt(
    json_path=str(expansion_json),
    prompt_specs=[MASK_PROMPT_SPEC],
    name="mask_prompt",
)
sam3_mask = SAM3MaskPropagation(max_tracker_states=5, name="sam3_mask")
prop_overlay = TrackingOverlayNode(alpha=0.4, name="tracking_overlay")
prop_video = ToVideoNode(
    output_video_path=str(mask_video_path), frame_rate=FRAME_RATE, name="to_video"
)

prop_pipeline.connect(
    (pr_cu3s.outputs.cube, pr_rgb.cube),
    (pr_cu3s.outputs.wavelengths, pr_rgb.wavelengths),
    (pr_rgb.rgb_image, sam3_mask.inputs.rgb_frame),
    (pr_cu3s.outputs.mesu_index, sam3_mask.inputs.frame_id),
    (pr_cu3s.outputs.mesu_index, mask_prompt.inputs.frame_id),
    (mask_prompt.outputs.mask, sam3_mask.inputs.mask),
    (pr_rgb.rgb_image, prop_overlay.rgb_image),
    (pr_cu3s.outputs.mesu_index, prop_overlay.frame_id),
    (sam3_mask.outputs.mask, prop_overlay.mask),
    (sam3_mask.outputs.object_ids, prop_overlay.object_ids),
    (prop_overlay.rgb_with_overlay, prop_video.rgb_image),
    (pr_cu3s.outputs.mesu_index, prop_video.frame_id),
)

prop_dm = Cu3sDataModule(
    cu3s_file_path=str(cu3s_path),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    measurement_indices=list(range(PROMPT_FRAME_ID, PROMPT_FRAME_ID + NUM_FRAMES)),
)
prop_pipeline.to(device)
Predictor(pipeline=prop_pipeline, datamodule=prop_dm).predict(collect_outputs=False)

if not mask_video_path.exists():
    raise RuntimeError(f"Mask-propagation video was not created: {mask_video_path}")
logger.success("Mask-propagation overlay: {}", mask_video_path)

In [ ]:
display(Video(str(mask_video_path), embed=IN_COLAB, width=640))

## 6 - Free GPU memory

In [ ]:
sam3_point.cleanup()
sam3_mask.cleanup()
if torch.cuda.is_available():
    torch.cuda.empty_cache()